# 05 — LLM / SLM Pipeline Experiments

Benchmark three locally-hosted SLMs via Ollama:

| Model | Size | Quant |
|-------|------|-------|
| `phi4:14b-q4_K_M` | 14B | Q4_K_M |
| `mistral:7b` | 7B | Q4 |
| `llama3.1:8b` | 8B | Q4 |

Evaluated on:
- Detection accuracy (vs ground-truth labels)
- ATT&CK technique mapping accuracy
- LLM-as-judge quality scores
- Latency (p50 / p95)

Dataset: LMD-2023 (windowed, 200 test windows)

In [ ]:
import sys
sys.path.insert(0, r'h:\Challenge-3-4\cyber-anomaly-detection')

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# Build test windows from LMD-2023 (full dataset)
from data.ingest.lmd2023 import load_full
from pipeline_classic.features.windowing import build_sliding_windows_with_labels

df_lmd = load_full(variant='2.3M')

# Use raw text DataFrame for LLM (not encoded features)
# Build window indices manually
WINDOW_SIZE = 32
STRIDE = 16
MAX_TEST_WINDOWS = 200   # LLM inference limit — each window = 1 API call per model

windows_dfs = []
windows_labels = []

for start in range(0, len(df_lmd) - WINDOW_SIZE, STRIDE):
    chunk = df_lmd.iloc[start:start + WINDOW_SIZE]
    label = int((chunk['label'].fillna(0).astype(int) > 0).any())
    windows_dfs.append(chunk)
    windows_labels.append(label)
    if len(windows_dfs) >= MAX_TEST_WINDOWS:
        break

print(f'Test windows: {len(windows_dfs)}  anomaly rate: {np.mean(windows_labels):.3f}')

In [ ]:
# Helper: run one SLM model on all test windows
from pipeline_llm.slm.ollama_client import OllamaClient
from pipeline_llm.formatter import build_prompt, messages_for_ollama, parse_slm_response
from pipeline_llm.rag.retriever import get_context
from evaluation.metrics import detection_metrics, LatencyTracker

def benchmark_model(model_name: str, windows, labels, use_rag: bool = True):
    client = OllamaClient(model=model_name)
    tracker = LatencyTracker()
    preds, scores, results = [], [], []

    for i, (wdf, lbl) in enumerate(zip(windows, labels)):
        try:
            rag_ctx = get_context(wdf) if use_rag else ''
            prompt = build_prompt(wdf, rag_ctx)
            msgs = messages_for_ollama(prompt)

            t0 = time.perf_counter()
            raw = client.infer(msgs)
            latency = (time.perf_counter() - t0) * 1000
            tracker.record(latency)

            res = parse_slm_response(raw.get('content', '{}'))
            detected = int(bool(res.get('anomaly_detected', False)))
            conf = float(res.get('anomaly_score', 0.5))

            preds.append(detected)
            scores.append(conf)
            results.append({'window': i, 'label': lbl, 'pred': detected,
                            'score': conf, 'latency_ms': latency,
                            'techniques': res.get('techniques', []),
                            'explanation': res.get('explanation', '')})
        except Exception as exc:
            print(f'  Window {i} failed: {exc}')
            preds.append(0); scores.append(0.0)
            results.append({'window': i, 'label': lbl, 'pred': 0, 'score': 0.0,
                            'latency_ms': 0.0, 'techniques': [], 'explanation': ''})

        if (i + 1) % 20 == 0:
            print(f'  [{model_name}] {i+1}/{len(windows)} done')

    metrics = detection_metrics(np.array(labels), np.array(preds), np.array(scores))
    lat_stats = tracker.stats()
    return metrics, lat_stats, results

print('Helper defined.')

In [ ]:
# ---- Run benchmarks ----
# NOTE: Make sure Ollama is running and models are pulled:
#   ollama pull phi4:14b-q4_K_M
#   ollama pull mistral:7b
#   ollama pull llama3.1:8b

MODELS_TO_TEST = [
    'phi4:14b-q4_K_M',
    'mistral:7b',
    'llama3.1:8b',
]

all_results = {}

for model_name in MODELS_TO_TEST:
    print(f'\n=== Benchmarking {model_name} ===')
    try:
        m, lat, res = benchmark_model(model_name, windows_dfs, windows_labels)
        all_results[model_name] = {'metrics': m, 'latency': lat, 'details': res}
        print(f'  F1={m["f1"]:.3f}  AUROC={m["roc_auc"]:.3f}'
              f'  p50={lat["p50_ms"]:.0f}ms  p95={lat["p95_ms"]:.0f}ms')
    except Exception as exc:
        print(f'  SKIPPED ({exc})')

In [ ]:
# Summary table
rows = []
for model, data in all_results.items():
    m, lat = data['metrics'], data['latency']
    rows.append({
        'model': model,
        'precision': round(m.get('precision', 0), 3),
        'recall':    round(m.get('recall', 0), 3),
        'f1':        round(m.get('f1', 0), 3),
        'roc_auc':   round(m.get('roc_auc', 0), 3),
        'p50_ms':    round(lat.get('p50_ms', 0), 1),
        'p95_ms':    round(lat.get('p95_ms', 0), 1),
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

# Save to JSON for use in compare.py
with open('llm_benchmark_results.json', 'w') as f:
    json.dump({k: {'metrics': v['metrics'], 'latency': v['latency']} for k, v in all_results.items()}, f, indent=2, default=float)
print('\nSaved to llm_benchmark_results.json')

In [ ]:
if rows:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Detection metrics
    summary_df.set_index('model')[['precision','recall','f1','roc_auc']].plot(
        kind='bar', ax=axes[0], rot=15
    )
    axes[0].set_title('SLM Detection Metrics')
    axes[0].set_ylabel('Score')
    axes[0].legend(loc='lower right')

    # Latency
    summary_df.set_index('model')[['p50_ms','p95_ms']].plot(
        kind='bar', ax=axes[1], rot=15, color=['steelblue','tomato']
    )
    axes[1].set_title('SLM Latency (ms)')
    axes[1].set_ylabel('ms')

    plt.tight_layout()
    plt.show()

## LLM-as-Judge Evaluation (optional)

Requires `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` in environment.

In [ ]:
RUN_JUDGE = False   # Set True when API keys are available

if RUN_JUDGE and all_results:
    from pipeline_llm.judge.scorer import batch_score

    # Take first model's results
    first_model = list(all_results.keys())[0]
    details = all_results[first_model]['details']

    # Build records for judging (sample 20 anomaly windows)
    judge_records = [
        {
            'events_summary': f'Window {r["window"]}: {len(windows_dfs[r["window"]])} events',
            'slm_output': str({'anomaly_detected': r['pred'], 'techniques': r['techniques'],
                               'explanation': r['explanation']}),
            'ground_truth': {'is_anomaly': bool(r['label'])},
        }
        for r in details if r['label'] == 1
    ][:20]

    judge_results = batch_score(judge_records)

    judge_df = pd.DataFrame(judge_results)
    print(judge_df[['detection_accuracy','technique_mapping','explanation_quality',
                    'false_positive_risk','overall_score']].describe())
else:
    print('Judge evaluation skipped (set RUN_JUDGE=True to enable).')